# Evaluation, calibration, profit, and group diagnostics

Separate rank performance, probability accuracy, decision economics, and group outcomes.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
import pandas as pd

from creditriskbook.data.datasets import load_dataset
from creditriskbook.decisioning import cutoff_table
from creditriskbook.models import evaluate_pd, fit_pd_model, score_pd, split_dataset

bundle = load_dataset("synthetic_retail", n_rows=7_000, seed=404)
train, test = split_dataset(bundle, bundle.frame)
model = fit_pd_model(bundle, train)
predicted_pd = score_pd(model, test)
metrics = evaluate_pd(test[bundle.target], predicted_pd)
policies = cutoff_table(predicted_pd, test[bundle.target].to_numpy())
best = policies.loc[policies["realised_profit"].idxmax()]
print(metrics)
print(best.to_dict())

In [ ]:
audit = test[["sex", "age", bundle.target]].copy()
audit["approved"] = predicted_pd < best["pd_cutoff"]
group = audit.groupby("sex").agg(
    observations=("approved", "size"),
    approval_rate=("approved", "mean"),
    observed_default_rate=(bundle.target, "mean"),
)
group["approval_rate_ratio_to_max"] = group["approval_rate"] / group["approval_rate"].max()
print(group)
assert group["observations"].sum() == len(test)

A disparity diagnostic is a question, not a legal conclusion. Investigate sample size, label bias, legitimate need, alternative specifications, intersectional groups, uncertainty, and applicable law with qualified reviewers.